# Step 9: LightGBM Global Model Backtest

In this notebook, we train and evaluate a global LightGBM model on the tabular features engineered using Spark.

### Why one model per store instead of one model across all 30k series?
- 10 stores × ~940 days (d >= 1000) × ~30 features ≈ 30M rows, which risks running out of RAM in local pandas.
- A single store (`CA_1`) has ~2.8M rows. Training one model per store is still a true **global model**: it learns joint patterns across all 3,049 items in that store.
- Per-store and per-department models were among the top-performing strategies in the original M5 competition.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd

from src.backtest import split_fold
from src.config import CUTOFFS, DEV_FOLDS, EXPORTS, HOLDOUT, SEED, TRAIN_START_D
from src.metrics import coverage, eligible_ids, pinball, wape, wrmsse

STORES = ["CA_1", "CA_2", "CA_3", "CA_4", "TX_1", "TX_2", "TX_3", "WI_1", "WI_2", "WI_3"]
CATS = ["item_id", "dept_id", "cat_id", "event_type_1"]
FEATURES = ([
    "lag_28", "lag_35", "lag_42", "lag_364",
    "rmean_7", "rmean_28", "rmean_56", "rstd_28",
    "dow", "dom", "month", "snap", "is_event",
    "sell_price", "price_norm", "price_change_7d", "price_rel_dept"
] + CATS)

print("Configuration and modules loaded successfully.")
print(f"Selected {len(FEATURES)} features: {FEATURES}")

In [ ]:
# Load scale and weight metadata for scoring
stats = pd.read_parquet(EXPORTS / "series_stats")
print(f"Loaded series stats with shape: {stats.shape}")

def load_store(store: str) -> pd.DataFrame:
    """Loads the feature table for a single store and optimizes memory usage.
    
    - Converts categorical columns to 'category' before train/test split to freeze category codes.
    - Casts float64 numerical columns to float32 to reduce memory footprint by ~50%.
    """
    df = pd.read_parquet(EXPORTS / "features" / f"store_id={store}")
    for c in CATS:
        df[c] = df[c].astype("category")  # Category codes fixed before train/test split
    num = df.select_dtypes("float64").columns
    df[num] = df[num].astype("float32")  # Downcast float64 to float32
    return df

# Test loading store CA_1
df_ca1 = load_store("CA_1")
print(f"CA_1 loaded successfully: {len(df_ca1):,} rows, {df_ca1.shape[1]} columns")
print(f"Memory usage: {df_ca1.memory_usage().sum() / 1e6:.1f} MB")